# Walk through the book

Same functions as the page. Nothing here downloads prices or writes the cache.

The window below is Start `2026-01-02`, End `2026-10-05`.


In [1]:
%reload_ext autoreload
%autoreload 2

import importlib
import os
from pathlib import Path

import pandas as pd

import src.data
import src.pnl

importlib.reload(src.data)
importlib.reload(src.pnl)

from src.data import PAIRS, load_portfolio, load_prices
from src.pnl import (
    _balances,
    _book_history,
    _day_changes,
    _dollar_cross,
    _dollar_results,
    _exposure_path,
    _exposure_table,
    _last_complete_close,
    _move,
    _position_rows,
    _position_values,
    _ranked_day,
    _same_pnl,
    _split_sum,
    _usable_var_window,
    _window_history,
    _window_pnl,
    build_report,
    historical_var,
    position_value,
    usd_per_currency,
)

here = Path.cwd()
if (here / "assignment" / "data" / "portfolio.csv").exists():
    os.chdir(here / "assignment")
elif not (here / "data" / "portfolio.csv").exists():
    raise FileNotFoundError("Run this notebook from the assignment folder.")

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 20)

start = pd.Timestamp("2026-01-02")
as_of = pd.Timestamp("2026-10-05")
book = load_portfolio()
prices = load_prices().loc[:as_of]
book


,trade_id,trade_date,currency_pair,notional_base,entry_price
0,T1,2026-01-02,USDJPY,10000000,156.7310
1,T2,2026-02-16,EURUSD,-5000000,1.1867
2,T3,2026-03-30,USDSGD,-8000000,1.2898
3,T4,2026-05-11,USDINR,6000000,94.4253
4,T5,2026-06-22,USDKRW,-4000000,"1,531.3300"
5,T6,2026-08-03,USDCNY,5000000,6.7509
6,T7,2026-09-15,AUDUSD,7000000,0.7138


## Closes

The last two cached rows. The page marks the book on the second of these.


In [2]:
prices.tail(2)


,EURUSD,AUDUSD,USDJPY,USDSGD,USDCNY,USDINR,USDKRW
date,,,,,,,
2026-10-02,1.1250,0.6929,157.9270,1.2802,6.7045,96.2248,"1,360.5900"
2026-10-05,1.1206,0.6957,157.6750,1.2802,6.6973,96.2800,"1,343.8101"


## USD per currency

`usd_per_currency`. USD is 1. EUR and AUD use the close. JPY, SGD, CNY, INR, and KRW use 1/close.


In [3]:
rates = usd_per_currency(prices)
jpy_close = prices.at[as_of, "USDJPY"]
eur_close = prices.at[as_of, "EURUSD"]
print("JPY close", jpy_close)
print("1 / JPY close", 1 / jpy_close)
print("rate JPY", rates.at[as_of, "JPY"])
print("EUR close", eur_close)
print("rate EUR", rates.at[as_of, "EUR"])
rates.loc[[start, as_of]]


JPY close 157.6750030517578
1 / JPY close 0.006342159382560745
rate JPY 0.006342159382560745
EUR close 1.1205737590789795
rate EUR 1.1205737590789795


,USD,EUR,AUD,JPY,SGD,CNY,INR,KRW
date,,,,,,,,
2026-01-02,1.0000,1.1750,0.6678,0.0064,0.7784,0.1429,0.0111,0.0007
2026-10-05,1.0000,1.1206,0.6957,0.0063,0.7811,0.1493,0.0104,0.0007


## One trade's dollar value

`position_value` is `foreign_amount * (dollars_now - dollars_then)`.

The first two lines are the hand checks: long 1,000,000 EUR at 1.10 marked at 1.12, then long 10,000,000 USDJPY at 150 marked at 155. The table under them is the book on the end date. That column is inception P&L.


In [4]:
print("long EUR", position_value(1_000_000, 1.10, 1.12))
print("long USDJPY", position_value(-10_000_000 * 150, 1 / 150, 1 / 155))

values = _position_values(book, rates)
values.loc[values["date"] == as_of, ["currency_pair", "value"]].set_index("currency_pair")


long EUR 20000.00000000002
long USDJPY 322580.64516129094


,value
currency_pair,
USDJPY,"59,870.0001"
EURUSD,"330,863.5950"
USDSGD,"60,304.4425"
USDINR,"115,581.5250"
USDKRW,"558,173.8171"
USDCNY,"-40,016.0026"
AUDUSD,"-126,901.2690"


## Day P&L and window P&L

`_day_changes` is today's value minus yesterday's. The pair's first row has no previous value, so that day's P&L is the value itself.

`_window_pnl` is the end value minus the start value. A pair whose first row is after the start contributes its end value.


In [5]:
day = _day_changes(values)
rows = []
pairs = []
for pair in book["currency_pair"]:
    if pair not in pairs:
        pairs.append(pair)
for pair in pairs:
    held = values.loc[values["currency_pair"] == pair].set_index("date")["value"]
    day_pnl = day.loc[(day["currency_pair"] == pair) & (day["date"] == as_of), "day_pnl"].iloc[0]
    rows.append({
        "currency_pair": pair,
        "day_pnl_usd": day_pnl,
        "window_pnl_usd": _window_pnl(held, start, as_of),
        "inception_pnl_usd": held.at[as_of],
    })
pnl = pd.DataFrame(rows).set_index("currency_pair")
pnl.loc["sum"] = pnl.sum()
pnl


,day_pnl_usd,window_pnl_usd,inception_pnl_usd
currency_pair,,,
USDJPY,"-15,861.1372","59,870.0001","59,870.0001"
EURUSD,"22,060.8711","330,863.5950","330,863.5950"
USDSGD,126.0946,"60,304.4425","60,304.4425"
USDINR,"3,375.5546","115,581.5250","115,581.5250"
USDKRW,"56,215.1241","558,173.8171","558,173.8171"
USDCNY,"-5,412.6824","-40,016.0026","-40,016.0026"
AUDUSD,"19,474.7448","-126,901.2690","-126,901.2690"
sum,"79,978.5697","957,876.1081","957,876.1081"


## Cross and Dollar

`_dollar_cross`. Each foreign currency held yesterday gets one vote. Dollar is yesterday's foreign net times that average. Cross is what is left. A balance opened today adds nothing, because fills are dealt at the close. The two columns add up to day P&L.


In [6]:
quantities = _balances(book, rates.index)
split = _dollar_cross(quantities, rates)
today = split.loc[as_of]
print("cross", today["cross_pnl_usd"])
print("dollar", today["dollar_pnl_usd"])
print("cross + dollar", today["cross_pnl_usd"] + today["dollar_pnl_usd"])
print("day pnl", pnl.loc["sum", "day_pnl_usd"])


cross 98997.82400576265
dollar -19019.254350723786
cross + dollar 79978.56965503887
day pnl 79978.56965503967


## VaR

`historical_var` takes today's currency balances and multiplies the foreign-currency dollars by the last 252 daily moves. A day with any missing move is left out. Portfolio VaR is the 5% worst loss of the days that remain. Marginal VaR is the component divided by the absolute foreign-leg dollars, not by the USD size. That dollar amount is today's balance times today's dollar price.


In [7]:
var = historical_var(quantities, rates, as_of)
print("portfolio var", var["portfolio_var_usd"])
print("var date", var["var_date"].date())
print("rank", var["var_rank"], "of", var["var_days"])
print("worst day", var["stress_date"].date(), var["stress_pnl_usd"])

live = rates.loc[as_of]
legs = []
for trade in book.itertuples(index=False):
    base, quote = PAIRS[trade.currency_pair]
    currency = base if quote == "USD" else quote
    legs.append({
        "currency_pair": trade.currency_pair,
        "foreign": currency,
        "foreign_usd": float(quantities.at[as_of, currency]) * float(live[currency]),
    })
var["by_pair"].merge(pd.DataFrame(legs), on="currency_pair")


portfolio var 99361.14853649819
var date 2026-01-26
rank 13 of 252
worst day 2025-04-07 -349217.66093854455


,currency_pair,position_var_usd,component_var_usd,marginal_var,worst_day_pnl_usd,foreign,foreign_usd
0,USDJPY,"79,105.8755","213,644.7431",0.0215,"-54,542.4014",JPY,"-9,940,129.9999"
1,EURUSD,"35,139.7860","49,498.8693",0.0088,"30,033.8449",EUR,"-5,602,868.7954"
2,USDSGD,"29,223.7066","-56,943.3453",-0.0071,"-68,114.6559",SGD,"8,060,304.4425"
3,USDINR,"41,121.3725","3,684.7474",0.0006,"16,952.2543",INR,"-5,884,418.4750"
4,USDKRW,"46,946.6950","-58,207.0175",-0.0128,"-24,501.4991",KRW,"4,558,173.8171"
5,USDCNY,"12,592.1758",0.0000,0.0000,"9,552.2705",CNY,"-5,040,016.0026"
6,AUDUSD,"43,767.0279","-52,316.8485",-0.0107,"-258,597.4742",AUD,"4,869,565.3081"


## Currency exposure

`_exposure_table` on the end date. `day_usd_return` is the percent move in USD per unit. It does not use the position size. Net is the sum of `usd_value`, and it equals inception P&L.


In [8]:
exposure = _exposure_table(quantities, rates, as_of).copy()
returns = []
for value in exposure["day_usd_return"]:
    if pd.isna(value):
        returns.append("")
    else:
        returns.append(f"{value:.2%}")
exposure["day_usd_return"] = returns
print("net", exposure["usd_value"].sum())
exposure


net 957876.1080971248


,currency,balance,usd_per_unit,usd_value,day_usd_return
1,JPY,"-1,567,310,028.0762",0.0063,"-9,940,129.9999",0.16%
0,USD,"9,937,265.8134",1.0000,"9,937,265.8134",0.00%
3,SGD,"10,318,559.6466",0.7811,"8,060,304.4425",0.00%
4,INR,"-566,551,803.5889",0.0104,"-5,884,418.4750",-0.06%
2,EUR,"-5,000,000.0000",1.1206,"-5,602,868.7954",-0.39%
6,CNY,"-33,754,498.9586",0.1493,"-5,040,016.0026",0.11%
7,AUD,"7,000,000.0000",0.6957,"4,869,565.3081",0.40%
5,KRW,"6,125,319,824.2188",0.0007,"4,558,173.8171",1.25%


## The other functions

These are called from the functions above. Each print is one of them on this same window.


In [9]:
history = _book_history(values, day)
print("_book_history on the end date")
print(history.loc[history["date"] == as_of])


results, signed = _dollar_results(quantities, rates, as_of)
print("_dollar_results signed foreign usd")
print(pd.Series(signed))
window = _usable_var_window(results)
print("_usable_var_window", len(window), window.index[0].date(), window.index[-1].date())

var_day, var_pnl = _ranked_day(window.sum(axis=1), 13)
stress_day, stress_pnl = _ranked_day(results.dropna(how="any").sum(axis=1), 1)
print("_ranked_day var", var_day.date(), -var_pnl)
print("_ranked_day worst", stress_day.date(), stress_pnl)

previous = rates.index[rates.index < as_of][-1]
print("_move JPY", previous.date(), _move(rates.at[as_of, "JPY"], rates.at[previous, "JPY"]))

path = _exposure_path(quantities, rates, start, as_of)
print("_exposure_path last row")
print(path.tail(1))

window_history = _window_history(history, start, as_of)
print("_window_history last row")
print(window_history.tail(1))

day_dollar, day_cross = _split_sum(split, [as_of])
life_dollar, life_cross = _split_sum(split, history["date"])
print("_split_sum day", day_cross, day_dollar)
print("_split_sum since inception", life_cross, life_dollar)
_same_pnl(float(history.loc[history["date"] == as_of, "day_pnl_usd"].iloc[0]), day_dollar, day_cross)
print("_same_pnl day ok")

print("_last_complete_close", _last_complete_close(book, prices, as_of).date())
print("_last_complete_close 2026-10-04", _last_complete_close(book, prices, "2026-10-04").date())

print("_position_rows")
_position_rows(book, prices, rates, values, day, var, start, as_of)


_book_history on the end date
          date  day_pnl_usd  cumulative_pnl_usd
196 2026-10-05  79,978.5697        957,876.1081
_dollar_results signed foreign usd
USDJPY   -9,940,129.9999
EURUSD   -5,602,868.7954
USDSGD    8,060,304.4425
USDINR   -5,884,418.4750
USDKRW    4,558,173.8171
USDCNY   -5,040,016.0026
AUDUSD    4,869,565.3081
dtype: float64
_usable_var_window 252 2025-10-15 2026-10-05
_ranked_day var 2026-01-26 99361.14853649819
_ranked_day worst 2025-04-07 -349217.66093854455
_move JPY 2026-10-02 0.0015982171967008352
_exposure_path last row
          date             JPY             EUR            SGD             INR  \
196 2026-10-05 -9,940,129.9999 -5,602,868.7954 8,060,304.4425 -5,884,418.4750   

               KRW             CNY            AUD            USD  \
196 4,558,173.8171 -5,040,016.0026 4,869,565.3081 9,937,265.8134   

            Foreign  
196 -8,979,389.7053  
_window_history last row
          date  day_pnl_usd  cumulative_pnl_usd  window_cumulative
195 202

,currency_pair,side,notional_base,usd_notional,entry_price,live_spot,day_pnl_usd,window_pnl_usd,inception_pnl_usd,position_var_usd,component_var_usd,marginal_var,worst_day_pnl_usd
0,USDJPY,Long USD,"10,000,000.0000","10,000,000.0000",156.7310,157.6750,"-15,861.1372","59,870.0001","59,870.0001","79,105.8755","213,644.7431",0.0215,"-54,542.4014"
1,EURUSD,Short EUR,"-5,000,000.0000","5,602,868.7954",1.1867,1.1206,"22,060.8711","330,863.5950","330,863.5950","35,139.7860","49,498.8693",0.0088,"30,033.8449"
3,USDINR,Long USD,"6,000,000.0000","6,000,000.0000",94.4253,96.2800,"3,375.5546","115,581.5250","115,581.5250","41,121.3725","3,684.7474",0.0006,"16,952.2543"
5,USDCNY,Long USD,"5,000,000.0000","5,000,000.0000",6.7509,6.6973,"-5,412.6824","-40,016.0026","-40,016.0026","12,592.1758",0.0000,0.0000,"9,552.2705"
6,AUDUSD,Long AUD,"7,000,000.0000","4,869,565.3081",0.7138,0.6957,"19,474.7448","-126,901.2690","-126,901.2690","43,767.0279","-52,316.8485",-0.0107,"-258,597.4742"
2,USDSGD,Short USD,"-8,000,000.0000","8,000,000.0000",1.2898,1.2802,126.0946,"60,304.4425","60,304.4425","29,223.7066","-56,943.3453",-0.0071,"-68,114.6559"
4,USDKRW,Short USD,"-4,000,000.0000","4,000,000.0000","1,531.3300","1,343.8101","56,215.1241","558,173.8171","558,173.8171","46,946.6950","-58,207.0175",-0.0128,"-24,501.4991"


## Page totals

`build_report` is what the page draws. The last cell checks these figures against the screen.


In [10]:
report = build_report(load_portfolio(), load_prices(), as_of, start)
fields = [
    "day_pnl_usd",
    "day_cross_pnl_usd",
    "day_dollar_pnl_usd",
    "window_pnl_usd",
    "inception_pnl_usd",
    "gross_usd",
    "portfolio_var_usd",
    "hedge_benefit_usd",
    "stress_pnl_usd",
]
for field in fields:
    print(f"{field:24} {report[field]:15,.2f}")
print("var_date                ", report["var_date"].date())
print("var_rank                ", report["var_rank"], "of", report["var_days"])
print("stress_date             ", report["stress_date"].date())

expected = {
    "day_pnl_usd": 79_979,
    "day_cross_pnl_usd": 98_998,
    "day_dollar_pnl_usd": -19_019,
    "inception_pnl_usd": 957_876,
    "window_pnl_usd": 957_876,
    "gross_usd": 43_472_434,
    "portfolio_var_usd": 99_361,
    "hedge_benefit_usd": 188_535,
}
for field, shown in expected.items():
    if round(report[field]) != shown:
        raise AssertionError(f"{field} is {report[field]}, screen shows {shown}")
if report["var_rank"] != 13 or report["var_days"] != 252:
    raise AssertionError("VaR rank is not 13 of 252")
if report["var_date"].date().isoformat() != "2026-01-26":
    raise AssertionError("VaR date is not 2026-01-26")
if report["stress_date"].date().isoformat() != "2025-04-07":
    raise AssertionError("worst day is not 2025-04-07")
print("matches the page")


day_pnl_usd                    79,978.57
day_cross_pnl_usd              98,997.82
day_dollar_pnl_usd            -19,019.25
window_pnl_usd                957,876.11
inception_pnl_usd             957,876.11
gross_usd                  43,472,434.10
portfolio_var_usd              99,361.15
hedge_benefit_usd             188,535.49
stress_pnl_usd               -349,217.66
var_date                 2026-01-26
var_rank                 13 of 252
stress_date              2025-04-07
matches the page
